# Montage from rendered PNG sequences

Build a montage from ground truth, Centralized S32, FedAvg S32, and FedProx S32 in that order. Source PNGs are kept at their original resolution; padding, holds, and transitions are explicit timeline operations. The FFV1 master is verified by RGBA round-trip comparison, while H.264 exports are viewing copies.

In [ ]:
from pathlib import Path
import os
import sys
from functools import partial

_start = Path(os.environ.get("BRATS_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
PROJECT_ROOT = next(
    (p for p in (_start, *_start.parents) if (p / "src" / "brats_pipeline").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Place notebooks/ and src/ under one project root, or set BRATS_PROJECT_ROOT.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
from brats_pipeline.common import ProjectPaths
paths_io = ProjectPaths(PROJECT_ROOT)
resolve_existing_path = paths_io.resolve
print("Project:", PROJECT_ROOT)

In [ ]:
from pathlib import Path
import json
import sys
import subprocess
from IPython.display import Video, display
from brats_pipeline.montage import (
    find_ffmpeg, discover_cases, show_cases, choose_folders, inspect_folders,
    get_canvas, show_endpoints, stamp, slug, write_json,
    build_plan, preview_pair, export_video, export_png_timeline,
    join_three_case_videos, MODES,
)

## Paths and timeline settings

In [ ]:
SOURCE_ROOT = PROJECT_ROOT / "docs/figures/brats_3d_cutaway_video"
OUTPUT_ROOT_12 = PROJECT_ROOT / "docs/figures/brats_3d_montage_12_from_png"
FFMPEG_BIN = "auto"
FPS = 12
REPEAT_EACH_FRAME = 1
HOLD_LAST_SECONDS = 8 / 12
START_HOLD_SECONDS = 0.0
FINAL_HOLD_SECONDS = 1.0
TRANSITION_SECONDS = 0.35
BLACK_GAP_SECONDS = 1 / 12
CANVAS_SIZE = None
ALLOW_DIFFERENT_SEQUENCES = False
VERIFY_LOSSLESS = True
MP4_CRF = 16
MP4_PRESET = "slow"

## Optional isolated FFmpeg installation

Use the isolated install only when the system and imageio FFmpeg binaries are unavailable. The wheel is installed into a separate cache directory.

In [ ]:
INSTALL_ISOLATED_FFMPEG = False
if INSTALL_ISOLATED_FFMPEG:
    target = Path.home() / ".cache/brats_montage_12/ffmpeg_wheel"
    target.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-deps", "--upgrade",
        "--target", str(target), "imageio-ffmpeg",
    ])

## Discover complete case groups

In [ ]:
FFMPEG_12 = find_ffmpeg(FFMPEG_BIN)
AVAILABLE_CASES_12 = discover_cases(SOURCE_ROOT) if SOURCE_ROOT.is_dir() else []
show_cases(AVAILABLE_CASES_12)

## Select one case and inspect source pixels

Set one of `CASE_INDEX`, `CASE_DIR` or `MANUAL_FRAMES_DIRS`. With multiple available cases, selection must be explicit.

In [ ]:
CASE_INDEX = None
CASE_DIR = None
MANUAL_FRAMES_DIRS = None

FRAME_DIRS_12, CASE_NAME_12 = choose_folders(
    case_index=CASE_INDEX, case_dir=CASE_DIR,
    manual_frames_dirs=MANUAL_FRAMES_DIRS, cases=AVAILABLE_CASES_12,
)
CLIPS_12 = inspect_folders(FRAME_DIRS_12, allow_different_sequences=ALLOW_DIFFERENT_SEQUENCES)
CANVAS_12 = get_canvas(CLIPS_12, CANVAS_SIZE)
RUN_DIR_12 = OUTPUT_ROOT_12 / slug(CASE_NAME_12) / stamp()
RUN_DIR_12.mkdir(parents=True, exist_ok=False)
write_json(RUN_DIR_12 / "sources.json", {"case": CASE_NAME_12, "canvas": CANVAS_12, "clips": CLIPS_12})
print("Canvas:", CANVAS_12)
print("Output:", RUN_DIR_12)

## Inspect clip endpoints

Only this diagnostic contact sheet uses thumbnails. Final montage frames retain the original pixel values.

In [ ]:
ENDPOINTS_12 = show_endpoints(CLIPS_12, CANVAS_12, RUN_DIR_12)

## Plan transitions

`cut` joins clips directly; `hold_cut` adds a final-frame hold. `fade_black`, `crossfade`, and `wipe` generate explicit transition frames with new blended pixels.

In [ ]:
def current_settings_12():
    return dict(
        fps=FPS, repeat_each=REPEAT_EACH_FRAME,
        hold_seconds=HOLD_LAST_SECONDS, transition_seconds=TRANSITION_SECONDS,
        black_seconds=BLACK_GAP_SECONDS, start_hold_seconds=START_HOLD_SECONDS,
        final_hold_seconds=FINAL_HOLD_SECONDS,
    )

for mode in MODES:
    plan = build_plan(CLIPS_12, mode, **current_settings_12())
    print(mode, plan["total_frames"], plan["seconds"])

## Short transition previews

In [ ]:
BUILD_PREVIEWS = False
PREVIEW_JOIN = 0
PREVIEW_CONTEXT_SECONDS = 1.5
PREVIEW_MODES = ["cut", "hold_cut", "fade_black", "crossfade", "wipe"]
PREVIEWS_12 = {}
if BUILD_PREVIEWS:
    pair = preview_pair(CLIPS_12, PREVIEW_JOIN, PREVIEW_CONTEXT_SECONDS, FPS)
    for mode in PREVIEW_MODES:
        plan = build_plan(pair, mode, **current_settings_12())
        output = export_video(clips=pair, canvas=CANVAS_12, plan=plan, ffmpeg=FFMPEG_12, out_dir=RUN_DIR_12 / "previews",
                              kind="viewing_mp4", verify=False, crf=MP4_CRF, preset=MP4_PRESET)
        PREVIEWS_12[mode] = output
        display(Video(str(output), embed=True, html_attributes="controls loop"))

## Export the final master and viewing copy

In [ ]:
FINAL_MODE = "hold_cut"
FINAL_FORMAT = "ffv1"  # Alternative: rgb_lossless_mp4 for fully opaque PNGs.
BUILD_FINAL = False
CREATE_VIEWING_MP4 = True

if BUILD_FINAL:
    if FINAL_FORMAT not in ("ffv1", "rgb_lossless_mp4"):
        raise ValueError("Choose a lossless master format.")
    FINAL_PLAN_12 = build_plan(CLIPS_12, FINAL_MODE, **current_settings_12())
    MASTER_12 = export_video(clips=CLIPS_12, canvas=CANVAS_12, plan=FINAL_PLAN_12, ffmpeg=FFMPEG_12, out_dir=RUN_DIR_12 / "masters",
                             kind=FINAL_FORMAT, verify=VERIFY_LOSSLESS)
    print("Master:", MASTER_12)
    if CREATE_VIEWING_MP4:
        VIEWING_12 = export_video(clips=CLIPS_12, canvas=CANVAS_12, plan=FINAL_PLAN_12, ffmpeg=FFMPEG_12, out_dir=RUN_DIR_12 / "exports",
                                  kind="viewing_mp4", verify=False, crf=MP4_CRF, preset=MP4_PRESET)
        display(Video(str(VIEWING_12), embed=True, html_attributes="controls loop"))

## Optional PNG timeline export

In [ ]:
EXPORT_PNG_TIMELINE = False
PNG_TIMELINE_MODE = "hold_cut"
if EXPORT_PNG_TIMELINE:
    plan = build_plan(CLIPS_12, PNG_TIMELINE_MODE, **current_settings_12())
    PNG_TIMELINE_DIR_12 = export_png_timeline(CLIPS_12, CANVAS_12, plan, RUN_DIR_12)

## Optional concatenation of three completed case montages

This joins three case videos, not three individual model clips. Compatible stream parameters, decoded frames and frame timestamps are checked. No new pauses or transitions are inserted.

In [ ]:
JOIN_ROOT = OUTPUT_ROOT_12
JOIN_KIND = "ffv1"
JOIN_INDICES = None

INSPECT_OR_JOIN_CASES = False
if INSPECT_OR_JOIN_CASES:
    JOINED_VIDEO = join_three_case_videos(
        root=JOIN_ROOT, kind=JOIN_KIND, indices=JOIN_INDICES, ffmpeg=FFMPEG_12,
    )